# Part 5: Cross-Encoder Rerank — 검색 결과 정밀 재정렬

이 섹션에서는:
1. **이론**: Bi-Encoder vs Cross-Encoder의 구조적 차이
2. **왜 Reranking이 필요한가**: 1차 검색의 한계
3. **데모**: Rerank 전후 문서 순위·점수 변화를 테이블로 시각화
4. **비용과 지연 시간**: 실측 데이터 확인

## 5-1. Bi-Encoder vs Cross-Encoder

RAG에서 사용하는 두 가지 유사도 모델 구조가 있습니다:

**Bi-Encoder (FAISS 등에서 사용)**
```
질문 → [Encoder] → 질문 벡터 ─┐
                                ├─ 코사인 유사도 → 점수
문서 → [Encoder] → 문서 벡터 ─┘
```
- 질문과 문서를 **독립적으로** 인코딩
- 매우 빠름 (문서 벡터를 미리 계산해두면 검색은 O(1))
- 질문-문서 간 **교차 어텐션**이 없어 세밀한 관련성 파악에 한계

**Cross-Encoder (Reranker에서 사용)**
```
질문 + 문서 → [Encoder] → 관련성 점수
```
- 질문과 문서를 **함께** 입력하여 교차 어텐션 수행
- 훨씬 정밀한 관련성 점수 산출
- 느림 (문서마다 별도 추론 필요 → top_k=20이면 20번 추론)

## 5-2. 왜 Reranking이 필요한가

1차 검색(FAISS/BM25/Ensemble)은 **속도 우선**으로 설계되어 있습니다.
넓은 범위에서 후보 문서(top_k=20)를 빠르게 가져오지만, 순위가 완벽하지 않습니다.

**2단계 검색 전략 (Retrieve → Rerank)**:
```
전체 문서 (수천~수만 개)
    │
    ├── 1단계: Retrieve (FAISS/BM25) → 후보 20개 (빠름)
    │
    └── 2단계: Rerank (Cross-Encoder) → 최종 5개 (정밀)
```

- 1단계에서 **재현율(Recall)** 확보: 관련 문서를 놓치지 않도록 넓게 검색
- 2단계에서 **정밀도(Precision)** 향상: 진짜 관련 있는 문서만 상위로
- LLM에 전달되는 맥락의 **품질**이 올라가므로 답변 정확도 향상

In [1]:
# 패키지 임포트 및 데이터 로드
import pickle
import time
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv()

with open("splits_v2.pkl", "rb") as f:
    splits_data = pickle.load(f)
splits = splits_data["vlm"]

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

RAG_PROMPT = """다음 맥락을 바탕으로 질문에 답하세요. 맥락에 없는 내용은 추측하지 마세요.

맥락:
{context}

질문: {question}
답변:"""

print(f"VLM 추출 청크 수: {len(splits)}")

/var/folders/48/477vrv_n08b185fbd1ycjrqh0000gn/T/ipykernel_21614/4238677113.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


VLM 추출 청크 수: 105


In [2]:
# Cross-Encoder 모델 로드
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('BAAI/bge-reranker-v2-m3')
print("bge-reranker-v2-m3 로드 완료")

bge-reranker-v2-m3 로드 완료


## 5-3. Rerank 데모: 검색 → 점수화 → 재정렬

1단계로 Ensemble Retriever에서 20개 후보를 가져온 뒤,
Cross-Encoder로 각 문서의 관련성 점수를 매기고 재정렬합니다.
**Rerank 전후 순위가 어떻게 바뀌는지** 관찰합니다.

In [11]:
# 1단계: Ensemble Retriever로 후보 20개 검색
TOP_K = 20
TOP_N = 6

vs = FAISS.from_documents(splits, embeddings)
faiss_r = vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})
bm25_r = BM25Retriever.from_documents(documents=splits, k=TOP_K)
retriever = EnsembleRetriever(retrievers=[faiss_r, bm25_r], weights=[0.5, 0.5])

demo_question = "삼성전자 DS 부문 2025년 4분기 영업이익은?"
candidate_docs = retriever.invoke(demo_question)
print(f"1단계 검색 결과: {len(candidate_docs)}개 후보 문서")

1단계 검색 결과: 26개 후보 문서


In [12]:
# 2단계: Cross-Encoder로 각 문서에 관련성 점수 부여
t0 = time.time()
pairs = [[demo_question, doc.page_content] for doc in candidate_docs]
scores = cross_encoder.predict(pairs)
rerank_time = time.time() - t0

scored_docs = list(zip(scores, range(len(candidate_docs)), candidate_docs))
scored_docs_sorted = sorted(scored_docs, key=lambda x: x[0], reverse=True)

print(f"Rerank 소요 시간: {rerank_time:.2f}s ({len(candidate_docs)}개 문서)")
print(f"\n{'='*80}")
print(f" Rerank 전후 순위 변화 (상위 {TOP_N}개)")
print(f"{'='*80}")
print(f"{'Rerank후':>8} {'Rerank전':>8} {'점수':>8}  {'내용 (앞 80자)'}")
print(f"{'-'*8:>8} {'-'*8:>8} {'-'*8:>8}  {'-'*50}")

for new_rank, (score, old_rank, doc) in enumerate(scored_docs_sorted[:TOP_N]):
    page = doc.metadata.get('page', '?')
    snippet = doc.page_content.strip().replace('\n', ' ')
    max_chars = 120  # 더 많은 컨텐츠를 보여주기 위해 최대 표시 글자수 확장
    wrapped = [snippet[j:j+60] for j in range(0, min(len(snippet), max_chars), 60)]
    rank_change = old_rank - new_rank
    arrow = '↑' if rank_change > 0 else ('↓' if rank_change < 0 else '─')
    print(f"  {new_rank+1:>4}위   {old_rank+1:>4}위  {score:>7.3f}  {arrow} (page {page})")
    for w in wrapped:
        print(f"       {w}")
    print("-"*80)

Rerank 소요 시간: 2.57s (26개 문서)

 Rerank 전후 순위 변화 (상위 6개)
 Rerank후  Rerank전       점수  내용 (앞 80자)
-------- -------- --------  --------------------------------------------------
     1위      1위    1.000  ─ (page 8)
       ```markdown ## 2025년 4분기 영업이익은 20.1조원  삼성전자의 2025년 4분기 영업이익은
        2025년 3분기 대비 65.4% 증가한 20.1조원이다. DS, 디스플레이를 제외한 나머지 사업부 영업이
--------------------------------------------------------------------------------
     2위      2위    1.000  ─ (page 7)
       ```markdown ## 2025년 4분기 매출액은 93.8조원  삼성전자의 2025년 4분기 매출액은 2
       025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3분기 대
--------------------------------------------------------------------------------
     3위      4위    1.000  ↑ (page 6)
       ```markdown # 삼성전자 (005930)  ## 메모리가 끌고 가는 실적  ### 25년 4분기, 
       압도적 메모리 실적  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조
--------------------------------------------------------------------------------
     4위      3위    0.999  ↓ (page 12)
       ```markdown ## 삼성전자/

In [13]:
# 전체 순위 변화 요약
print(f"\nRerank 전 Top-{TOP_N} 문서의 Rerank 후 순위:")
for i, (score, old_rank, doc) in enumerate(scored_docs[:TOP_N]):
    # scored_docs의 old_rank == i, 즉 rerank전에 i위였던 문서
    # scored_docs_sorted에서 orig_idx == old_rank일 때의 new_rank를 찾는다
    new_rank = next(j for j, (_, orig_idx, _) in enumerate(scored_docs_sorted) if orig_idx == old_rank)
    snippet = doc.page_content.strip().replace('\n', ' ')[:60]
    print(f"  Rerank 전 {i+1}위 → Rerank 후 {new_rank+1}위 (점수: {score:.3f}) | {snippet}...")

print(f"\n→ Cross-Encoder가 질문과의 관련성을 정밀하게 재평가하여 순위가 변동됩니다.")


Rerank 전 Top-6 문서의 Rerank 후 순위:
  Rerank 전 1위 → Rerank 후 1위 (점수: 1.000) | ```markdown ## 2025년 4분기 영업이익은 20.1조원  삼성전자의 2025년 4분기 영업이익은...
  Rerank 전 2위 → Rerank 후 2위 (점수: 1.000) | ```markdown ## 2025년 4분기 매출액은 93.8조원  삼성전자의 2025년 4분기 매출액은 2...
  Rerank 전 3위 → Rerank 후 4위 (점수: 0.999) | ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2026년 1분기 영업이익은...
  Rerank 전 4위 → Rerank 후 3위 (점수: 1.000) | ```markdown # 삼성전자 (005930)  ## 메모리가 끌고 가는 실적  ### 25년 4분기, ...
  Rerank 전 5위 → Rerank 후 15위 (점수: 0.797) | ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 25년 4분기는 SK하이닉스...
  Rerank 전 6위 → Rerank 후 17위 (점수: 0.763) | 4) VD/가전 사업부 매출액은 2025년 3분기 대비 6.1% 증가한 14.8조원이다. VD 매출액은 25...

→ Cross-Encoder가 질문과의 관련성을 정밀하게 재평가하여 순위가 변동됩니다.


## 5-4. top_n 파라미터와 정밀도-재현율 Trade-off

Rerank 후 **몇 개의 문서를 LLM에 전달할지** (top_n)는 중요한 하이퍼파라미터입니다:

| top_n | 정밀도 | 재현율 | LLM 비용 |
|-------|--------|--------|----------|
| 3 | 높음 | 낮을 수 있음 | 낮음 |
| 5 | 균형 | 균형 | 중간 |
| 10 | 낮을 수 있음 | 높음 | 높음 |

- **top_n이 작으면**: 정말 관련 있는 문서만 전달 → 정밀하지만 정보 부족 가능
- **top_n이 크면**: 많은 문서 전달 → 재현율은 높지만 노이즈도 증가, LLM 비용 증가

## 5-5. RAG 답변 비교: Rerank 적용 전 vs 후

In [6]:
expected_answer = "16.4조원"

# Rerank 전: 1단계 검색 결과에서 상위 5개만 사용
docs_before = candidate_docs[:TOP_N]

# Rerank 후: Cross-Encoder 점수 기준 상위 5개 사용
docs_after = [doc for _, _, doc in scored_docs_sorted[:TOP_N]]

def run_rag(docs, name):
    context = "\n---\n".join(d.page_content for d in docs)
    t0 = time.time()
    msg = llm.invoke([HumanMessage(content=RAG_PROMPT.format(context=context, question=demo_question))])
    elapsed = time.time() - t0
    answer = msg.content if hasattr(msg, "content") else str(msg)
    contains = expected_answer in answer
    print(f"\n[{name}]")
    print(f"  답변: {answer[:300]}")
    print(f"  정답 포함 여부: {'O' if contains else 'X'} (정답: {expected_answer})")
    print(f"  응답 시간: {elapsed:.2f}s")

run_rag(docs_before, "Rerank 전 (Ensemble Top-5)")
run_rag(docs_after, "Rerank 후 (Cross-Encoder Top-5)")


[Rerank 전 (Ensemble Top-5)]
  답변: 삼성전자 DS 부문 2025년 4분기 영업이익은 16.4조원입니다.
  정답 포함 여부: O (정답: 16.4조원)
  응답 시간: 1.09s

[Rerank 후 (Cross-Encoder Top-5)]
  답변: 삼성전자 DS 부문의 2025년 4분기 영업이익은 2025년 3분기 대비 134.4% 증가한 16.4조원입니다.
  정답 포함 여부: O (정답: 16.4조원)
  응답 시간: 1.13s


## 5-6. 비용과 지연 시간

Cross-Encoder Reranking의 실제 비용을 정리합니다.

In [7]:
# Rerank 시간 측정 (문서 수별)
for n in [5, 10, 20]:
    test_docs = candidate_docs[:n]
    test_pairs = [[demo_question, d.page_content] for d in test_docs]
    t0 = time.time()
    _ = cross_encoder.predict(test_pairs)
    elapsed = time.time() - t0
    print(f"문서 {n:>2}개 Rerank: {elapsed:.3f}s (문서당 {elapsed/n:.3f}s)")

print(f"\n참고: bge-reranker-v2-m3는 로컬 CPU에서 실행되므로 API 비용은 없습니다.")
print(f"GPU 사용 시 속도가 크게 향상됩니다.")

문서  5개 Rerank: 0.633s (문서당 0.127s)
문서 10개 Rerank: 0.853s (문서당 0.085s)
문서 20개 Rerank: 1.618s (문서당 0.081s)

참고: bge-reranker-v2-m3는 로컬 CPU에서 실행되므로 API 비용은 없습니다.
GPU 사용 시 속도가 크게 향상됩니다.


## 5-7. 정리

| 항목 | 설명 |
|------|------|
| **목적** | 1차 검색 결과를 정밀하게 재정렬하여 LLM에 전달할 맥락의 품질 향상 |
| **작동** | Cross-Encoder가 (질문, 문서) 쌍을 함께 입력받아 관련성 점수 산출 |
| **장점** | Bi-Encoder보다 훨씬 정밀한 관련성 판단, 정밀도(Precision) 향상 |
| **비용** | 문서 수만큼 Cross-Encoder 추론 필요 (CPU에서는 느림, GPU 권장) |
| **권장 설정** | top_k=20으로 넓게 검색 → Rerank → top_n=5로 압축 |

**전체 RAG 파이프라인 요약 (Part 2~5):**
```
PDF 추출 (Part 2)
  → 청킹 + 임베딩 인덱싱
    → 하이브리드 검색 (Part 3: FAISS + BM25)
      → 질문 확장 (Part 4: Multi-Query)
        → 정밀 재정렬 (Part 5: Cross-Encoder Rerank)
          → LLM 답변 생성
```
각 단계는 독립적으로 적용할 수도 있고, 조합하여 사용할 수도 있습니다.